In [ ]:
import pandas as pd
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

In [ ]:
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [ ]:
# tocanize the dataset
def tocanize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'", "")
  return text.split()

In [ ]:
tocanize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [ ]:
# form a vovubelery
vocab = {'<UNK>': 0}
type(vocab)

dict

In [ ]:
def build_vocab(row):
  tocanize_question = tocanize(row['question'])
  tocanize_ans = tocanize(row['answer'])

  marged_tocken = tocanize_question + tocanize_ans

  for tocken in marged_tocken :
    if tocken not in vocab :
      vocab[tocken] = len(vocab)

In [ ]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
len(vocab)

324

In [ ]:
# convert words to numerical indices
def text_to_indeces(sentence, vocab):
  indexed_text = []

  for tocken in tocanize(sentence):

    if tocken in vocab:
      indexed_text.append(vocab[tocken])

    else :
      indexed_text.append(vocab['<UNK>'])

  return indexed_text


In [ ]:
text_to_indeces('what is campusx', vocab)

[1, 2, 0]

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

class QA_dataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indeces(self.df.iloc[index]['question'], vocab)
    numerical_answer = text_to_indeces(self.df.iloc[index]['answer'], vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [ ]:
dataset = QA_dataset(df, vocab)

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
for que, ans in dataloader:
  print(que, ans)

tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([[54]])
tensor([[ 10,  11, 157, 158, 159]]) tensor([[160]])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([[238]])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([[205]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([[307]])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([[106]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[ 42,   2,   3, 210, 137, 168, 211, 169]]) tensor([[113]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([[194]])
tensor([[42, 18,  2, 62, 63,  3, 64, 18]]) tensor([[65]])
tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([[162]])
tensor([[ 42, 137, 118,   3, 247,   5, 248]]) tensor([[249]])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([[61]])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([

In [ ]:
import torch.nn as nn

In [ ]:
class simple_Rnn(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    self.embading = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64,batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self,question):
    embased_question = self.embading(question)
    hidden, final = self.rnn(embased_question)
    output = self.fc(final.squeeze(0))

    return output

In [ ]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [ ]:
embading = nn.Embedding(324, embedding_dim=50)

In [ ]:
a = embading(dataset[0][0])

In [ ]:
y = nn.RNN(50, 64)

In [ ]:
y(a)[1].shape

torch.Size([1, 64])

In [ ]:
learning_rate = 0.001
epochs = 20

In [ ]:
model = simple_Rnn(len(vocab))

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(epochs):
  total_loss = 0

  for que, ans in dataloader:

    optimizer.zero_grad()

    pred = model(que)

    loss1 = loss(pred, ans[0])

    loss1.backward()

    optimizer.step()

    total_loss = total_loss + loss1.item()

  print(f'{epoch+1}, {total_loss}')

1, 9.248759791254997
2, 8.13328218832612
3, 7.202909331768751
4, 6.449533607810736
5, 5.81130856461823
6, 5.209406530484557
7, 4.734187567606568
8, 4.326781345531344
9, 3.960807554423809
10, 3.5985665433108807
11, 3.314731892198324
12, 3.055410251021385
13, 2.820304064080119
14, 2.6118739461526275
15, 2.41631449572742
16, 2.2491251192986965
17, 2.0937986774370074
18, 1.9511209446936846
19, 1.8218827852979302
20, 1.7026322176679969


In [ ]:
def predict(model, que, thresh):
  numerical_question = text_to_indeces(que,vocab)

  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  output = model(question_tensor)

  prob = torch.nn.functional.softmax(output, dim=1)

  value, index = torch.max(prob, dim=1)

  if value < thresh :
    print('i dont know')


  return list(vocab.keys())[index]



In [ ]:
print(predict(model, '	Which superhero is also known as the Dark Knight', 0.5))

batman
